# Transfomer without Embeddings

## 1. Setup and Data Loading
Imports libraries and loads the protein dataset for K-fold cross-validation.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np
import math

# Load the dataset
df = pd.read_csv('/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-ss.cleaned.csv')

# Ensure there's a 'len' column with sequence lengths
if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()

# Pre-process sequences
df['seq'] = df['seq'].str.replace("*", "X") # Replace non-standard aa
df = df[df['has_nonstd_aa'] == False].reset_index(drop=True)

print(df.head())
df.info()
print(df.tail())

## 2. Create Vocabularies
**MODIFIED:** This cell replaces the ESM loader. We now create a vocabulary for the input amino acid sequences (`seq_vocab`) in addition to the label vocabularies.

In [ ]:
# Vocabularies for SST8 and SST3 labels
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

# NEW: Create vocabulary for input amino acid sequences
all_chars = set(''.join(df['seq']))
seq_vocab = {char: i+1 for i, char in enumerate(sorted(list(all_chars)))}
seq_vocab['<pad>'] = 0 # Add padding token
vocab_size = len(seq_vocab)

print(f"Sequence vocab size: {vocab_size}")
print(seq_vocab)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Define Dataset and Collate Function


In [ ]:
class ProteinSequenceDataset(Dataset):
    def __init__(self, sequences, sst8_labels, sst3_labels, seq_vocab, ss8_vocab, ss3_vocab):
        self.sequences = sequences
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels
        self.seq_vocab = seq_vocab
        self.ss8_vocab = ss8_vocab
        self.ss3_vocab = ss3_vocab

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        ss8 = self.sst8_labels[idx]
        ss3 = self.sst3_labels[idx]
        
        # Tokenize sequence
        seq_tokens = [self.seq_vocab.get(c, 0) for c in seq] # Default to <pad> if char not in vocab
        
        # Tokenize labels
        ss8_tokens = [self.ss8_vocab.get(c, -1) for c in ss8]
        ss3_tokens = [self.ss3_vocab.get(c, -1) for c in ss3]
        
        # Ensure label length matches sequence length
        ss8_tokens = ss8_tokens[:len(seq_tokens)]
        ss3_tokens = ss3_tokens[:len(seq_tokens)]
        
        return torch.tensor(seq_tokens, dtype=torch.long), torch.tensor(ss8_tokens, dtype=torch.long), torch.tensor(ss3_tokens, dtype=torch.long)

def collate_fn(batch):
    seqs, ss8s, ss3s = zip(*batch)
    
    # Pad sequences
    padded_seqs = pad_sequence(seqs, batch_first=True, padding_value=seq_vocab['<pad>'])
    
    # Pad labels (use -1 for padding, as in original code)
    padded_ss8s = pad_sequence(ss8s, batch_first=True, padding_value=-1)
    padded_ss3s = pad_sequence(ss3s, batch_first=True, padding_value=-1)
    
    return padded_seqs, padded_ss8s, padded_ss3s

## 4. Training Configuration with K-Fold Cross-Validation
**Training Configuration:**
- **K-Fold Cross-Validation**: 5 folds
- **Epochs**: 50 epochs per fold
- **Early Stopping**: Patience of 5 epochs (stops if validation accuracy doesn't improve)
- **Model**: Transformer encoder with learned embeddings (no pre-trained embeddings)


In [ ]:
# Model hyperparameters
embedding_dim = 128        
num_heads = 8             
num_layers = 4            
ff_dim = 512          
dropout = 0.1

# Training hyperparameters  
batch_size = 16
num_epochs = 50
learning_rate = 1e-4


## 5. K-Fold Cross-Validation Setup
Prepare data for K-fold cross-validation and create dataloaders.


In [ ]:
from sklearn.model_selection import KFold

# First, separate test set (10% of data)
all_indices = np.arange(len(df))
train_val_indices, test_indices = train_test_split(all_indices, test_size=0.1, random_state=42)

# Prepare test dataset
test_dataset = ProteinSequenceDataset(
    df.iloc[test_indices]['seq'].tolist(),
    df.iloc[test_indices]['sst8'].tolist(),
    df.iloc[test_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    collate_fn=collate_fn
)

# Set up 5-fold cross-validation on remaining data
n_folds = 5
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

print(f"Dataset sizes:")
print(f"  Training+Validation: {len(train_val_indices):,} samples")
print(f"  Test: {len(test_indices):,} samples")
print(f"  Using {n_folds}-fold cross-validation")


## 6. Define the Transformer Model
**Model Architecture:** Transformer encoder with learned embeddings and positional encoding. Takes token IDs as input (no pre-trained embeddings).

In [ ]:
class PositionalEncoding(nn.Module):
    """Standard Transformer Positional Encoding"""
    def __init__(self, d_model, dropout=0.1, max_len=6000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class ProteinTransformer(nn.Module):
    def __init__(self, vocab_size, input_dim=128, num_heads=8, num_layers=4, ff_dim=512, dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.embedding = nn.Embedding(vocab_size, input_dim, padding_idx=seq_vocab['<pad>'])
        self.pos_encoder = PositionalEncoding(input_dim, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Two separate classifier heads
        self.q8_head = nn.Linear(input_dim, 8)
        self.q3_head = nn.Linear(input_dim, 3)

    def forward(self, x, mask=None):
        """
        x: [batch_size, seq_len] (token IDs)
        mask: [batch_size, seq_len] (padding mask, True where padded)
        """
        x = self.embedding(x) * math.sqrt(self.input_dim)
        x = self.pos_encoder(x)
        
        x = self.transformer_encoder(x, src_key_padding_mask=mask)
        
        # Per-residue classification
        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)
        
        return q8_logits, q3_logits


## 7. Training Loop with K-Fold Cross-Validation and Early Stopping
**Training Configuration:**
- **K-Fold Cross-Validation**: 5 folds
- **Epochs**: 50 epochs per fold
- **Early Stopping**: Patience of 5 epochs (stops if validation accuracy doesn't improve)
- **Model**: Transformer encoder with learned embeddings (no pre-trained embeddings)


In [ ]:
def compute_accuracy(pred_logits, labels):
    """Per-residue accuracy ignoring -1 padding"""
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

# K-Fold Cross-Validation
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)

num_epochs = 50
patience = 5
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(train_val_indices)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{n_folds}")
    print(f"{'='*50}")
    
    # Get actual indices for this fold
    fold_train_indices = train_val_indices[train_idx]
    fold_val_indices = train_val_indices[val_idx]
    
    # Create datasets for this fold
    train_dataset = ProteinSequenceDataset(
        df.iloc[fold_train_indices]['seq'].tolist(),
        df.iloc[fold_train_indices]['sst8'].tolist(),
        df.iloc[fold_train_indices]['sst3'].tolist(),
        seq_vocab, ss8_vocab, ss3_vocab
    )
    val_dataset = ProteinSequenceDataset(
        df.iloc[fold_val_indices]['seq'].tolist(),
        df.iloc[fold_val_indices]['sst8'].tolist(),
        df.iloc[fold_val_indices]['sst3'].tolist(),
        seq_vocab, ss8_vocab, ss3_vocab
    )
    
    # Create dataloaders (without optimization flags)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    
    # Initialize model for this fold
    model = ProteinTransformer(vocab_size=vocab_size, input_dim=embedding_dim,
                              num_heads=num_heads, num_layers=num_layers,
                              ff_dim=ff_dim, dropout=dropout)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)
    
    # Optimizer for this fold
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    # Early stopping variables
    best_val_acc_q8 = 0.0
    epochs_no_improve = 0
    best_model_state = None
    
    # Training loop for this fold
    for epoch in range(num_epochs):
        model.train()
        train_loss, train_acc_q8, train_acc_q3 = 0, 0, 0
        
        for batch_idx, (seqs, ss8, ss3) in enumerate(tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{num_epochs}")):
            seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
            mask = (seqs == seq_vocab['<pad>'])
            
            # Forward pass (without mixed precision)
            q8_logits, q3_logits = model(seqs, mask)
            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            loss = loss_q8 + 0.5 * loss_q3
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_acc_q8 += compute_accuracy(q8_logits, ss8)
            train_acc_q3 += compute_accuracy(q3_logits, ss3)
        
        train_loss /= len(train_loader)
        train_acc_q8 /= len(train_loader)
        train_acc_q3 /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss, val_acc_q8, val_acc_q3 = 0, 0, 0
        with torch.no_grad():
            for seqs, ss8, ss3 in val_loader:
                seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
                mask = (seqs == seq_vocab['<pad>'])
                
                # Forward pass (without mixed precision)
                q8_logits, q3_logits = model(seqs, mask)
                loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
                loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
                loss = loss_q8 + 0.5 * loss_q3
                
                val_loss += loss.item()
                val_acc_q8 += compute_accuracy(q8_logits, ss8)
                val_acc_q3 += compute_accuracy(q3_logits, ss3)
        
        val_loss /= len(val_loader)
        val_acc_q8 /= len(val_loader)
        val_acc_q3 /= len(val_loader)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
        print(f"Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}")
        print(f"Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}")
        
        # Early stopping check
        if val_acc_q8 > best_val_acc_q8:
            best_val_acc_q8 = val_acc_q8
            epochs_no_improve = 0
            best_model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            print(f"✓ New best model! Val Acc Q8: {best_val_acc_q8:.4f}")
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s)")
            
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break
    
    # Save best model for this fold
    torch.save(best_model_state, f"best_scratch_transformer_model_fold{fold+1}.pt")
    fold_results.append({
        'fold': fold + 1,
        'best_val_acc_q8': best_val_acc_q8,
        'best_val_acc_q3': val_acc_q3
    })
    print(f"\nFold {fold+1} Best Val Acc Q8: {best_val_acc_q8:.4f}")

# Summary of all folds
print(f"\n{'='*50}")
print("K-FOLD CROSS-VALIDATION SUMMARY")
print(f"{'='*50}")
for result in fold_results:
    print(f"Fold {result['fold']}: Val Acc Q8 = {result['best_val_acc_q8']:.4f}")
avg_val_acc = np.mean([r['best_val_acc_q8'] for r in fold_results])
print(f"\nAverage Val Acc Q8 across all folds: {avg_val_acc:.4f}")

# Load best fold model for testing (highest validation accuracy)
best_fold = max(fold_results, key=lambda x: x['best_val_acc_q8'])
print(f"\nUsing model from Fold {best_fold['fold']} for final testing")


## 8. Final Evaluation on Test Set
Evaluates the best model from K-fold cross-validation on the held-out test set. Loads `best_scratch_transformer_model.pt`.

In [ ]:
# Initialize a new model instance and load the best fold's model
model = ProteinTransformer(
    vocab_size=vocab_size, 
    input_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    ff_dim=ff_dim,
    dropout=dropout
)
model.load_state_dict(torch.load(f"best_scratch_transformer_model_fold{best_fold['fold']}.pt"))

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.to(device)
model.eval()

test_loss, test_acc_q8, test_acc_q3 = 0, 0, 0
with torch.no_grad():
    for seqs, ss8, ss3 in tqdm(test_loader, desc="Testing"):
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        mask = (seqs == seq_vocab['<pad>'])
        
        # Forward pass (without mixed precision)
        q8_logits, q3_logits = model(seqs, mask)
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        
        test_loss += loss.item()
        test_acc_q8 += compute_accuracy(q8_logits, ss8)
        test_acc_q3 += compute_accuracy(q3_logits, ss3)

test_loss /= len(test_loader)
test_acc_q8 /= len(test_loader)
test_acc_q3 /= len(test_loader)

print(f"\n{'='*50}")
print("FINAL TEST SET EVALUATION")
print(f"{'='*50}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy Q8: {test_acc_q8:.4f}")
print(f"Test Accuracy Q3: {test_acc_q3:.4f}")
